# 01 — Data loading and first look

Goal: load the raw CrisisBench data and confirm it has the shape, columns, classes, and Nepal earthquake coverage we expect.

We'll go through:
1. Imports + path setup
2. Load the three JSON splits and concatenate them
3. Quick descriptive stats (shape, dtypes, null counts)
4. Class label distribution (raw counts)
5. Unique events and sources
6. Drill into the 2015 Nepal earthquake — sample tweets per category
7. Save the English-only DataFrame for downstream notebooks/scripts

## 1. Imports and path setup

We import the project's loader functions instead of repeating the JSON-parsing logic in every notebook. The `sys.path.insert` line tells Python that the project root is importable, so `from src.data_loader import ...` resolves correctly when the notebook is run from `notebooks/`.

In [1]:
# os, sys — make sure we can import from the project root
import os, sys
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# pandas — tabular data handling. df.head(), df.value_counts(), etc.
import pandas as pd

# Project-level helpers
from src.data_loader import (
    load_dataset,        # tries local JSON first, HuggingFace fallback
    filter_english,      # keeps only lang == 'en' rows
    save_processed,      # writes the cleaned CSV
    get_dataset_info,    # prints summary stats
    PROCESSED_CSV_PATH,
)

pd.set_option('display.max_colwidth', 140)  # show longer tweets in head() output

## 2. Load the raw dataset

`load_dataset()` returns a single DataFrame combining `train.json`, `dev.json`, and `test.json`, with an extra `split` column so we don't lose track of the official train/dev/test boundaries CrisisBench publishes.

In [2]:
df = load_dataset(prefer_local=True)
df.shape

[21:25:37] Loading dataset from local JSON files…
[21:25:37]   loading train.json
[21:25:37]     -> 99,073 rows
[21:25:37]   loading dev.json
[21:25:37]     -> 14,436 rows
[21:25:37]   loading test.json
[21:25:38]     -> 28,024 rows


(141533, 8)

## 3. First 10 rows + dtypes + null counts

Eyeball the raw data: are there obvious encoding issues, missing values, weird columns?

In [3]:
df.head(10)

,id,event,source,text,lang,lang_conf,class_label,split
0,207368210100662272,2012_italy_earthquakes,crisislext26,Anche questa volta è durata tanto..anche molto forte.. #Terremoto #Italia,it,1,other_relevant_information,train
1,18582,disaster_events,drd-figureeight-multimedia,Approximately 100km long firebreaks have been constructed by Indonesia and elsewhere.,en,1,infrastructure_and_utilities_damage,train
2,592616302138658817,2015_nepal_earthquake,crisisnlp-volunteers,God bless you... https://t.co/AnEy1ydkkz,en,NA,not_humanitarian,train
3,503643491143282688,2014_california_earthquake,crisisnlp-cf,"RT @perreaux: Cracked wine casks, damaged historical buildings and coffee shops. This Napa earthquake is the biggest first world disast...",en,NA,infrastructure_and_utilities_damage,train
4,323833109051228160,2013_boston_bombings-ontopic,crisislext6,I'm really just excited for new undies and pinkberry @mollymcnultzxo,en,1,not_humanitarian,train
5,508333923886067712,2014_pakistan_floods,crisisnlp-cf,"Rescue effort expands in India, Pakistan as flood death toll tops 350 http://t.co/SlbqlhIHmi http://t.co/8F2zjKZ95f #india #asia",en,1,injured_or_dead_people,train
6,451190348748845056,2014_chile_earthquake,crisisnlp-cf,RT @leanielsen: I hope everyone in Chile stays safe and are okay. Surrounding countries should watch out for the Tsunami alert. #PrayFor...,en,1,sympathy_and_support,train
7,541468991546331136,2014_philippines_typhoon-hagupit,crisisnlp-volunteers,"Typhoon howls through Philippines, more than 1 million flee http://t.co/i6ESgc8cPh",en,NA,not_humanitarian,train
8,497088043451699201,2014_worldwide_ebola,crisisnlp-cf,"RT @SebastienNemeth: #Liberia #monrovia Even for stories around #ebola treatment center,clean yourself and your stuff http://t.co/RmI8u...",en,1,disease_related,train
9,541950425184731136,2014_philippines_typhoon-hagupit,crisisnlp-volunteers,It���s a good thing that the government have done everything to avert any lost of lives from the onslaught of typhoon hagupit in the cou...,en,NA,not_humanitarian,train


In [4]:
df.dtypes

id             object
event          object
source         object
text           object
lang           object
lang_conf      object
class_label    object
split          object
dtype: object

In [5]:
# isna() returns a same-shape DataFrame of True/False; .sum() counts True per column.
df.isna().sum()

id             0
event          0
source         0
text           0
lang           0
lang_conf      0
class_label    0
split          0
dtype: int64

## 4. Class label distribution

How many tweets are in each category? This is the first warning that we have heavy class imbalance — `not_humanitarian` and `other_relevant_information` together dominate the dataset.

In [6]:
# Step 1: .value_counts() counts how many rows have each unique value.
# Step 2: pct = count / total * 100 turns counts into share-of-dataset percentages.
counts = df['class_label'].value_counts()
pct = (counts / len(df) * 100).round(2)
summary = pd.DataFrame({'count': counts, 'pct': pct})
summary

,count,pct
class_label,,
not_humanitarian,52939,37.40
other_relevant_information,44626,31.53
donation_and_volunteering,8350,5.90
requests_or_needs,7004,4.95
sympathy_and_support,6829,4.83
infrastructure_and_utilities_damage,5496,3.88
affected_individual,4211,2.98
caution_and_advice,3601,2.54
injured_or_dead_people,3168,2.24


## 5. Unique events, sources, and languages

How many distinct disaster events does CrisisBench cover, and which raw datasets were merged?

In [7]:
print('Unique events  :', df['event'].nunique())
print('Unique sources :', df['source'].nunique())
print('Unique langs   :', df['lang'].nunique())
print()
print('Events (first 100):')
for ev in sorted(df['event'].unique())[:100]:
    print(' ', ev)

Unique events  : 61
Unique sources : 9
Unique langs   : 77

Events (first 100):
  2011_joplin_tornado-a121571
  2011_joplin_tornado-a131709
  2012_colorado_wildfires
  2012_costa-rica_earthquake
  2012_guatemala_earthquake
  2012_italy_earthquakes
  2012_philipinnes_floods
  2012_philippines_typhoon-pablo
  2012_sandy_hurricane-ontopic
  2012_us_sandy-hurricane-a143145
  2012_us_sandy-hurricane-a144267
  2012_venezuela_refinery-explosion
  2013_alberta_floods
  2013_alberta_floods-ontopic
  2013_australia_bushfire
  2013_bangladesh_savar-building-collapse
  2013_bohol_earthquake
  2013_boston_bombings
  2013_boston_bombings-ontopic
  2013_brazil_nightclub-fire
  2013_canada_lac-megantic-train-crash
  2013_colorado_floods
  2013_glasgow_helicopter-crash
  2013_italy_sardinia
  2013_la_airport-shootings
  2013_manila_floods
  2013_ny_train-crash
  2013_oklahoma_tornado-ontopic
  2013_pakistan_earthquake
  2013_phillipines_typhoon-yolanda
  2013_queensland_floods
  2013_queensland_floods-

In [8]:
df['source'].value_counts()

source
crisislext6                   49806
crisisnlp-volunteers          21680
crisislext26                  19893
crisisnlp-cf                  18394
crisismmd                     16020
drd-figureeight-multimedia     7419
aidr_system                    6116
iscram13                       1506
swdm13                          699
Name: count, dtype: int64

In [9]:
# Top 10 languages by row count. CrisisBench is dominated by English but
# Spanish, Italian, French, Portuguese all show up in non-trivial amounts.
df['lang'].value_counts().head(10)

lang
en    132619
es      4879
it      1337
fr       799
tl       745
pt       527
ru       123
id        52
hi        52
co        36
Name: count, dtype: int64

## 6. Nepal earthquake spotlight

Nepal 2015 is one of the headline events in this dataset. We pull out those tweets, count classes within the event, and show one or two sample tweets per category.

In [10]:
# Substring match on the event name — CrisisBench tags some events with year
# prefixes and -ontopic / -offtopic suffixes.
nepal_mask = df['event'].str.contains('nepal_earthquake', case=False, na=False)
# print(f'nepal_earthquake: {nepal_mask}')
nepal = df[nepal_mask]
print(f'Nepal earthquake tweets: {len(nepal):,}')
nepal['class_label'].value_counts()

Nepal earthquake tweets: 11,032


class_label
not_humanitarian                       5795
other_relevant_information             1343
donation_and_volunteering               963
sympathy_and_support                    888
response_efforts                        855
injured_or_dead_people                  399
infrastructure_and_utilities_damage     310
missing_and_found_people                233
requests_or_needs                        92
displaced_and_evacuations                86
caution_and_advice                       40
personal_update                          28
Name: count, dtype: int64

In [11]:
# Show 1 sample tweet per Nepal category — useful for understanding what each
# label actually looks like on real text.
for label, group in nepal.groupby('class_label'):
    print(f'\n--- {label} ({len(group)} tweets) ---')
    sample = group.sample(min(2, len(group)), random_state=42)
    for _, row in sample.iterrows():
        print('  •', row['text'][:200])


--- caution_and_advice (40 tweets) ---
  • listening on 20m 14.210 Emergency traffic from Nepal please keep radio frequency clear #hamradio #NepalEarthquake
  • another tremor. #NepalQuake

--- displaced_and_evacuations (86 tweets) ---
  • #HindustanTimes Indians flee earthquake-devastated Nepal, leaving behind jobs and savings http://t.co/vfnLw8mZLk‰Û_ http://t.co/GRjbZVVLzL
  • ((Noticias SIN)) The Latest on Nepal Quake: Thousands Try to Leave Kathmandu: Thousands line up at bus stations in Kathmandu in bid t...

--- donation_and_volunteering (963 tweets) ---
  • RT @AnneFHughes: Nepal needs help.  I have been personally involved with @dzifoundation. I trust them completely.  Please donate. https://t‰Û_
  • RT @tajinderbagga: Telecom Minister RS prasad order BSNL to Charge all call to Nepal as Local call #IndiaWithNepal

--- infrastructure_and_utilities_damage (310 tweets) ---
  • Patan durbar square is in ruins. Famous temples, including the Krishna temple reportedly destroyed. #Ka

## 7. Filter to English and save

Our preprocessing pipeline (lemmatiser, stopwords) is English-specific. Mixing languages in would dilute quality, so we drop non-English rows here and persist the result for the EDA and modelling notebooks.

In [12]:
df_en = filter_english(df)
save_processed(df_en)
df_en.shape

[21:25:39]   filter_english: kept 132,619 / dropped 8,914
[21:25:39]   wrote 132,619 rows to /Users/prabinupreti/work/projects/MLCourseworkCrisisDataSet/data/processed/crisisbench_en.csv


(132619, 8)

In [13]:
# Sanity check: the saved CSV exists and round-trips cleanly.
reloaded = pd.read_csv(PROCESSED_CSV_PATH)
print('Saved CSV    :', PROCESSED_CSV_PATH)
print('Reloaded shape:', reloaded.shape)
print('Splits       :')
print(reloaded['split'].value_counts())

Saved CSV    : /Users/prabinupreti/work/projects/MLCourseworkCrisisDataSet/data/processed/crisisbench_en.csv
Reloaded shape: (132619, 8)
Splits       :
split
train    92757
test     26294
dev      13568
Name: count, dtype: int64


## What we learned

- The dataset has the seven columns we expected (`id, event, source, text, lang, lang_conf, class_label`) plus the `split` column we added.
- Class imbalance is severe: `not_humanitarian` alone is ~37% of the data, and the smallest classes (e.g. `terrorism_related`) have only a handful of tweets. **This already tells us we cannot rely on plain accuracy as a metric — see the EDA and modelling notebooks for the full discussion.**
- Nepal earthquake coverage spans most humanitarian categories, so it makes a useful focused case study later in the project.
- We have ~133k English-only tweets after filtering — enough to train classical models comfortably without subsampling.